In [2]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


import matplotlib
# matplotlib.use('Agg')
import matplotlib.pyplot as plt
import copy
import numpy as np
from torchvision import datasets, transforms
import torch

from utils.sampling import mnist_iid, mnist_noniid, cifar_iid
#from utils.options import args_parser
from models.Update import LocalUpdate
from models.Nets import MLP, CNNMnist, CNNCifar, LeNet, CNNMnist2,MobileNetV2
from models.Fed import FedAvg
from models.Fed import FedQAvg, Quantization, Quantization_Finite, my_score, my_score_Finite
from models.test import test_img
import math
import random
import torch.nn as nn


ModuleNotFoundError: No module named 'utils'

In [ ]:

# In[2]:


class my_argument:    
    epochs = 1000    #"rounds of training"
    alpha = 0.1
    num_users = 6  # "number of users: K"
    frac = 0.5 #"the fraction of clients: C"
    local_ep=5 #"the number of local epochs: E"
    local_bs=10 #"local batch size: B"
    bs=128 #"test batch size"
    lr=0.001 #"learning rate"
    momentum=0.5 # "SGD momentum (default: 0.5)"
    weight_decay=0
    split='user' # "train-test split type, user or sample"
    
    #quantization
    f_size=32
    q=20

    # model arguments
    #model = 'mobilenetv2'
    model = 'cnn'
    kernel_num=9 #, help='number of each kind of kernel')
    kernel_sizes='3,4,5' #  help='comma-separated kernel size to use for convolution')
    norm='batch_norm' #, help="batch_norm, layer_norm, or None")
    num_filters=32 #, help="number of filters for conv nets")
    max_pool='True' #help="Whether use max pooling rather than strided convolutions")

    # other arguments
    dataset='mnist' #, help="name of dataset")
    iid=0
    num_classes=10#, help="number of classes")
    num_channels=1#, help="number of channels of images")
    gpu=3#, help="GPU ID, -1 for CPU")
    stopping_rounds=10#, help='rounds of early stopping')
    verbose='False'#, help='verbose print')
    opt='SGD'
    seed=1#, help='random seed (default: 1)')
    
args = my_argument()


In [ ]:
# In[3]:
def weights_init(m):
    if isinstance(m, nn.Conv2d):
        seed=123
        torch.cuda.manual_seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)
        random.seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        nn.init.xavier_uniform(m.weight.data, nn.init.calculate_gain('relu'))
        #nn.init.xavier_uniform(m.bias.data)
        torch.nn.init.zeros_(m.bias.data)
    if isinstance(m, torch.nn.Linear):
        seed=123
        torch.cuda.manual_seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)
        random.seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.nn.init.xavier_uniform_(m.weight.data)
        #torch.nn.init.xavier_uniform_(m.bias.data)
        torch.nn.init.zeros_(m.bias.data)
    if isinstance(m, nn.BatchNorm2d):
        seed=123
        torch.cuda.manual_seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)
        random.seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.nn.init.xavier_uniform_(m.weight.data)
        #torch.nn.init.xavier_uniform_(m.bias.data)
        torch.nn.init.zeros_(m.bias.data)
        #conv1.bias.data.fill_(0.01)
        

In [ ]:



def phiQ(p,sc, q, w):
    w_cap = w[:,0]
    #w_cap=[item[0] if isinstance(item,list) and len(item)>0 else item for item in w_cap]
    #w_cap=np.array(w_cap)
    #print("w_cap_first")
    #print(w_cap)
    w_cap= sc*w_cap
    v=np.floor(q*w_cap)
    one=np.ones(len(w))
    r=np.random.uniform(0,1,len(w_cap))
    temp=(one.T+np.sign(q*w_cap-v-r))*np.sign(q*w_cap-v-r)
    #j=np.sign(w_cap-v-r)
    #print("jjj")
    #print(j)
    #temp= (1/q)*(1/2)*temp
    temp=(1/2)*temp
    #print("temp")
    #print(temp)
    #w_cap= (1/q)*v + temp
    w_cap=(1/q)*(v+temp)
    w_cap=q*w_cap
    #w_cap=w_cap+ (1/2)*p*(-np.sign(w_cap)+one.T)*(-np.sign(w_cap))
    w_cap=w_cap+ (1/2)*(p-5)*(-np.sign(w_cap)+one.T)*(-np.sign(w_cap))
    #print("w_cap_last")
    #print(w_cap)
    del temp
    del one
    del v
    del r
    return w_cap
args.device = torch.device('cuda:{}'.format(args.gpu) if torch.cuda.is_available() and args.gpu != -1 else 'cpu')

# load dataset and split users
if args.dataset == 'mnist':
    trans_mnist = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    dataset_train = datasets.MNIST('/data/mnist/', train=True, download=True, transform=trans_mnist)
    dataset_test = datasets.MNIST('/data/mnist/', train=False, download=True, transform=trans_mnist)
    # sample users
    if args.iid:
        dict_users = mnist_iid(dataset_train, args.num_users)
        print('iid dataset')
    else:
        dict_users = mnist_noniid(dataset_train, args.num_users)
        print("non iid dataset")
elif args.dataset == 'cifar':
    trans_cifar = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    dataset_train = datasets.CIFAR10('/data/cifar', train=True, download=True, transform=trans_cifar)
    dataset_test = datasets.CIFAR10('/data/cifar', train=False, download=True, transform=trans_cifar)
    if args.iid:
        dict_users = cifar_iid(dataset_train, args.num_users)
    else:
        exit('Error: only consider IID setting in CIFAR10')
else:
    exit('Error: unrecognized dataset')
img_size = dataset_train[0][0].shape


# In[4]:


use_cuda = torch.cuda.is_available()
#use_cuda = torch.cuda.is_available()
#print(use_cuda)
args.device = torch.device("cuda" if use_cuda else "cpu")
print(use_cuda)
#args.device = torch.device("cpu")
print(args.device)





from models.Fed import FedAdd,FedSubstract,weight_vectorization_gen,FedAvg_gradient
import numpy as np
import copy
# build model
if args.model == 'cnn' and args.dataset == 'cifar':
    net_glob = CNNCifar(args=args).to(args.device)
elif args.model == 'cnn' and args.dataset == 'mnist':
    net_glob = CNNMnist(args=args).to(args.device)
elif args.model == "mobilenetv2":
    net_glob = MobileNetV2(args=args).to(args.device)
elif args.model == 'mlp':
    len_in = 1
    for x in img_size:
        len_in *= x
    net_glob = MLP(dim_in=len_in, dim_hidden=200, dim_out=args.num_classes).to(args.device)
else:
    exit('Error: unrecognized model')
print(net_glob)
net_glob.apply(weights_init)
net_glob.train()

# copy weights
w_glob = net_glob.state_dict()
#print(w_glob)
# for k in w_glob.keys():
#     print(k)
#     print(w_glob[k].shape)
abs_vect,layer_size=weight_vectorization_gen(w_glob)
print(abs_vect)


# # 1. FedAvg

# In[6]:



# training
loss_train = []
loss_test_arr = []
acc_test_arr = []
err_no_array=[]
update_no_array=[]
cv_loss, cv_acc = [], []
val_loss_pre, counter = 0, 0
net_best = None
best_loss = None
val_acc_list, net_list = [], []
user_no=args.num_users
m_local=[]
d=len(abs_vect)
print(d)
part=5000
N=args.num_users
segment=math.floor(d/part)
remain=d%part
#K_local=round(sparsity*d)
#m_global=np.zeros((d,1))
iter_no=args.epochs
avg=[]
comb=[]
idxs_users=[]
dropout=0
survive=1-dropout
m=round(survive*args.num_users)
for i in range(user_no):
    idxs_users.append(i)
idxs_users = np.random.choice(range(args.num_users), m, replace=False)
#idxs_users=idxs_users[0:m]

spar_level=[]
group=3
frac=int((args.num_users)/group)

BW=[0.0625,0.1875,0.25,0.3125,0.375]
BW=[0.03125, 0.09,0.09,0.125,0.125]
BW=[0.0625,0.09375,0.125,0.156,0.1875]
BW=[0.0625,0.09375,0.125,0.125,0.25]
BW=[4/32,20/32,24/32]
#BW=[0.1,0.2,0.3,0.4,0.5]
#BW=[0.0625,0.125,0.156,0.1875,0.2]
#BW=[1,1,1,1,1]
spar_level=[]
sum1=0
j=0
N=args.num_users
for i in range(args.num_users):
    spar_level.append(BW[j])
    sum1=sum1+1
    if sum1==frac:
       j=j+1
       sum1=0
print(spar_level)
product=1
x=N-1
prob=[]
for i in range(N-1):
    z=((1-spar_level[i])/product)**(1/x)
    prob.append(1-z)
    product=product*z
    x=x-1

sparsity=prob
print(sparsity)
mask=[]
mask_store=[]
for i in range(N):
    mask_store.append(np.zeros(d))
for i in range(N):
    #mask_store[i]=np.zeros(d)
    for j in range(N):
        if (i !=j) & (i<j):
            K_local=round(sparsity[i]*d)
            location_local=np.random.choice(range(d),K_local,replace=False)
            mask=np.zeros(d)
            np.put(mask,location_local,1)
            mask_store[i]=np.sign(mask_store[i]+mask)
            mask_store[j]=np.sign(mask_store[j]+mask)

    
for i in range(iter_no):
    comb.append([])
m=round(survive*args.num_users)
for iter in range(iter_no): #args.epochs
    print("iteration no.",iter)
    
    w_locals, loss_locals,diff_locals,grad_locals,grad_locals2, m_local = [], [],[],[],[],[]
    #m = 100
    #idxs_users = np.random.choice(range(args.num_users), m, replace=False)
    
    #print(len(idxs_users))
    updated=[]
    model_diff=[]
    grad_vect=[]
    prev=[]
    error=[]
    grad_vect_quant=[]
    grad_vect_quant2=[]
    grad_vect_send=[]
    grad_vect_send2=[]
    store_grad=[]
    location_local=[]
    abs_vect,layer_size=weight_vectorization_gen(w_glob) # vectorizing the global gradient
    mask=[]
    mask_store=[]
    for i in range(N):
         mask_store.append(np.zeros(d))
    for i in range(N):
    #mask_store[i]=np.zeros(d)
         for j in range(N):
             if (i !=j) & (i<j):
                K_local=round(sparsity[i]*d)
                location_local=np.random.choice(range(d),K_local,replace=False)
                mask=np.zeros(d)
                np.put(mask,location_local,1)
                mask_store[i]=np.sign(mask_store[i]+mask)
                mask_store[j]=np.sign(mask_store[j]+mask)
                
    idxs_users = np.random.choice(range(args.num_users), m, replace=False)
    for i in range(N):
        updated.append([])
        model_diff.append([])
        grad_vect.append([])
        prev.append([])
        #error.append(np.zeros((d,1)))
        grad_vect_send.append([])
        grad_vect_send2.append([])
        grad_vect_quant.append([])
        grad_vect_quant2.append([])
        #m_local.append(np.zeros((d,1)))
        store_grad.append([])
        #location_local.append([])
    for user in idxs_users:
        #print(user)
        updated[user]=copy.deepcopy(w_glob)
        local = LocalUpdate(args=args, dataset=dataset_train, idxs=dict_users[user])
        w, loss = local.train(net=copy.deepcopy(net_glob).to(args.device))
        prev[user]=updated[user]
        model_diff[user]=FedSubstract(w,prev[user])
        grad_vect[user],layer_size=weight_vectorization_gen(model_diff[user]) # vectorizing the gradient
        
        #location_local[user]=np.random.choice(range(d),K_local,replace=False)
        scale=1/((args.num_users)*spar_level[user])
        #print(user)
        
        count=0
        grad_vect_quant[user]=phiQ(np.power(2,args.f_size),scale,2**args.q,grad_vect[user])
        
        
            
        #grad_vect_quant[user]=phiQ(np.power(2,args.f_size),scale,2**args.q,grad_vect[user])
        #print(user)
        #mask=np.zeros(d)
        #np.put(mask,location_local[user],1)
        mask=mask_store[user]
        #print("user")
        #print(user)
        #print("length of sparsified gradient")
        #print(len(mask[np.nonzero(mask)]))
        #mask=np.random.choice([1, 0], size=d, p=[sparsity, 1-sparsity])
            #calculating the modified gradient to be sent to server
        grad_vect_send[user]=np.multiply(mask,grad_vect_quant[user])
        #grad_vect_send2[user]=np.multiply(mask,grad_vect_quant2[user])
        
        grad_locals.append(grad_vect_send[user])
        #grad_locals2.append(grad_vect_send2[user])
        #store_grad[user]=grad_vect_send[user]
        
        #w_locals.append(copy.deepcopy(w))
        loss_locals.append(copy.deepcopy(loss))
        #diff_locals.append(copy.deepcopy(model_diff[user]))
    #m_global_prev=m_global
    #m_global=np.zeros((d,1))
    del updated
    del prev
    del model_diff
    del mask
    del mask_store
    grad_avg=np.nan_to_num(sum(grad_locals)) # taking the average of masked gradients
    #grad_avg2=np.nan_to_num(sum(grad_locals2))
    #print("grad_avg")
    #print(grad_avg)
    grad_avg_correct = np.zeros_like(grad_avg)
    #print("before modulo")
    #print(grad_avg)
    grad_avg= (grad_avg)%(np.power(2,args.f_size)-5)
    #print("after modulo")
    #print(grad_avg)
    p=np.power(2,args.f_size)-5
    for i in range(len(grad_avg)):
        if grad_avg[i]>=0 and grad_avg[i]<(p-1)/2:
                        # print("Valid")
            grad_avg_correct[i]=grad_avg[i]
            grad_avg_correct[i]=(1/(2**args.q))*grad_avg_correct[i]
            #grad_avg_correct[i]=(1/(2**args.q))*grad_avg_correct[i]
            continue
        elif grad_avg[i]>=(p-1)/2 and grad_avg[i]<p:
                        # print("Chenged")
            grad_avg_correct[i]=grad_avg[i]-p
            grad_avg_correct[i]=(1/(2**args.q))*grad_avg_correct[i]
    #grad_avg_correct=np.nan_to_num(grad_avg)
    #print("after converting to real field")
    #print(grad_avg_correct)
    #print("overlap error")
    #error=grad_avg2-grad_avg_correct
    #print(error)
    #number of error locations
    #err_no=np.where(abs(error)>10**(-4))
    #print("number of error locations")
    #print(len(err_no[0]))
    #err_no_array.append(len(err_no[0]))
    #number of updated locs
    #update_no=np.nonzero(grad_avg_correct)
    #update_no_array.append(len(update_no[0]))
    #print("number of updated locations")
    #print(len(update_no[0]))
    
        #grad_avg_correct[i]=grad_avg[i]
    #grad_avg_correct=grad_avg
    #print("grad_avg_correct")
    #print(grad_avg_correct)
    count=0
    #abs_vect=abs(grad_avg)
    #Overlapping counts
    location=[]
    #grad_avg_correct=np.nan_to_num(grad_avg_correct)
    
    
    count=0
    w_glob_prev=copy.deepcopy(w_glob)
    flat=[]
    #conver
    for i in range(len(w_glob.keys())): # 4 layers in parameter
        flat.append([])
#     for h in w_glob_prev.keys():
#             if(count==0)|(count==2): #means if key in input_layer weight or hidden_layer_weight
#                 z=layer_size[count][0]*layer_size[count][1]
#             else: #means key is input_bias or hidden_bias
#                 z=layer_size[count][0]
                    
#             flat[count]=grad_avg[0:z] # taking out the vector for the specified layer
#             grad_avg=np.delete(grad_avg,np.s_[0:z])# deleting that vector from decoded after taking out
#             if(count==0)|(count==2):
#                 new=flat[count].reshape(layer_size[count][0],layer_size[count][1]) #reshaping back to the marix
#             else:
#                 new=flat[count]    # for bias, no need to reshape. keep it as a vector   
#             w_glob[h]=torch.from_numpy(new) #converting the matrix to a tensor
#             #print(w_glob[cluster_no][h].shape)
#             count=count+1

    for h in w_glob_prev.keys():
        s=list(w_glob[h].shape)
        if (len(s)==0):
            new=np.array(0)
            grad_avg_correct=np.delete(grad_avg_correct,np.s_[0])
        else:
            z=np.prod(list(w_glob[h].shape))
            flat[count]=grad_avg_correct[0:z] # taking out the vector for the specified layer
            grad_avg_correct=np.delete(grad_avg_correct,np.s_[0:z])# deleting that vector from decoded after taking out
             
            new=flat[count].reshape(list(w_glob[h].shape)) #reshaping back to the marix
              
        w_glob[h]=torch.from_numpy(new) #converting the matrix to a tensor
            #print(w_glob[cluster_no][h].shape)
        count=count+1
    # update global weights
    global_diff = w_glob
    #print(w_glob)
    w_glob=FedAdd(w_glob_prev,global_diff)
    

    # copy weight to net_glob
    net_glob.load_state_dict(w_glob)
    
    del w_glob_prev
    del grad_avg
    del flat
    torch.cuda.empty_cache()
    del grad_avg_correct
    del grad_vect_quant
    del grad_vect_send
    # print loss
    loss_avg = np.nan_to_num(sum(loss_locals) / len(loss_locals))
    
    loss_train.append(float(loss_avg))
    
    acc_test, loss_test = test_img(net_glob, dataset_test, args)
    acc_test_arr.append(float(acc_test))
    loss_test_arr.append(loss_test)
    if iter % 1 ==0:
        print('Round {:3d}, Average loss {:.3f} Test accuracy {:.3f}'.format(iter, loss_avg,acc_test))
    #print(loss_train)
    if iter%50==0:
        print("accuracy array")
        print(acc_test_arr)
        print("train loss")
        print(loss_train)
    


# In[ ]:


print(range(d))


# In[7]:


train_loss=[]
for i in loss_train:
    train_loss.append(float(i))
print('train_loss')
print(train_loss)


# In[8]:


test_acc=[]
for i in acc_test_arr:
    test_acc.append(float(i))
print('test_acc')
print(test_acc)



# In[ ]:




